# 02b_optimizacion · Tiempos antes y después (A2.6)
**Grupo 03.** Tres técnicas, cada una medida 3 veces (se reporta la mediana).
1. **Broadcast join** frente a sort-merge (hints de Spark SQL).
2. **Particionamiento** por año frente a tabla sin particionar, en una consulta filtrada por fecha.
3. **OPTIMIZE + Z-ORDER** por `Store`, en consulta filtrada por tienda.

El caché (`.cache()`) no se usa: en cómputo serverless de Free Edition no está soportado.

In [0]:
import time, statistics
from pyspark.sql import functions as F
ESQUEMA = "workspace.default"
resultados = []

def medir(tecnica, variante, fn, n=3):
    tiempos = []
    for _ in range(n):
        t0 = time.time(); fn(); tiempos.append(time.time() - t0)
    resultados.append((tecnica, variante, round(statistics.median(tiempos), 2), round(min(tiempos), 2), round(max(tiempos), 2)))
    print(tecnica, variante, "mediana", round(statistics.median(tiempos), 2), "s")

## Técnica 1 · Broadcast join

In [0]:
def consulta_join(hint):
    return lambda: spark.sql(f"""
      SELECT /*+ {hint}(s) */ s.Type, SUM(v.Weekly_Sales) AS venta
      FROM {ESQUEMA}.plata_ventas v JOIN {ESQUEMA}.plata_stores s ON v.Store = s.Store
      GROUP BY s.Type""").collect()

medir("1. join", "antes: sort-merge", consulta_join("MERGE"))
medir("1. join", "después: broadcast", consulta_join("BROADCAST"))

1. join antes: sort-merge mediana 1.24 s
1. join después: broadcast mediana 1.2 s


## Técnica 2 · Particionamiento por año

In [0]:
base = spark.table(f"{ESQUEMA}.plata_ventas").withColumn("anio", F.year("Date"))
base.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{ESQUEMA}.opt_ventas_plana")
base.write.format("delta").mode("overwrite").option("overwriteSchema", True).partitionBy("anio").saveAsTable(f"{ESQUEMA}.opt_ventas_part")

def consulta_anio(tabla):
    return lambda: spark.sql(f"SELECT Store, SUM(Weekly_Sales) FROM {ESQUEMA}.{tabla} WHERE anio = 2012 GROUP BY Store").collect()

medir("2. partición", "antes: sin particionar", consulta_anio("opt_ventas_plana"))
medir("2. partición", "después: partitionBy(anio)", consulta_anio("opt_ventas_part"))

2. partición antes: sin particionar mediana 0.84 s
2. partición después: partitionBy(anio) mediana 0.73 s


## Técnica 3 · OPTIMIZE + Z-ORDER

In [0]:
def consulta_tienda():
    return spark.sql(f"SELECT Dept, SUM(Weekly_Sales) FROM {ESQUEMA}.opt_ventas_plana WHERE Store = 20 GROUP BY Dept").collect()

medir("3. Z-ORDER", "antes", consulta_tienda)
spark.sql(f"OPTIMIZE {ESQUEMA}.opt_ventas_plana ZORDER BY (Store)")
medir("3. Z-ORDER", "después", consulta_tienda)

3. Z-ORDER antes mediana 0.77 s
3. Z-ORDER después mediana 0.76 s


In [0]:
tabla = spark.createDataFrame(resultados, ["tecnica", "variante", "mediana_s", "min_s", "max_s"])
display(tabla)
tabla.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{ESQUEMA}.opt_resultados_tiempos")

tecnica,variante,mediana_s,min_s,max_s
1. join,antes: sort-merge,1.24,1.19,2.81
1. join,después: broadcast,1.2,1.05,1.21
2. partición,antes: sin particionar,0.84,0.74,0.9
2. partición,después: partitionBy(anio),0.73,0.71,0.91
3. Z-ORDER,antes,0.77,0.75,0.84
3. Z-ORDER,después,0.76,0.7,1.62


**Lectura honesta.** Con ~420 mil filas, las diferencias pueden ser pequeñas o incluso favorecer la variante "antes":
el costo fijo de Spark domina. Reportar lo medido y explicarlo; no forzar una mejora que los números no muestran.